## Pip

In [ ]:
# !pip install seaborn

## import

In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path

import utils

## Utils

In [ ]:
BASE_DIR = Path.cwd()
RESULTS_PATH = BASE_DIR / "results"

## Datas

#### user

In [ ]:
e_12, m_12 = utils.get_pickle_day(12)
e_19, m_19 = utils.get_pickle_day(19)

In [ ]:
df_meta_12 = pd.DataFrame.from_dict(m_12, orient="index").reset_index(drop=True)
df_meta_19 = pd.DataFrame.from_dict(m_19, orient="index").reset_index(drop=True)

In [ ]:
df_meta_12.columns = [
    "id_12",
    "age_12",
    "sexe_12",
    "Big_number_12",
    "comportement_12",
    "cellule_home_12",
    "cellule_work_12",
    "nb_records_12",
    "nb_moves_12",
]
df_meta_19.columns = [
    "id_19",
    "age_19",
    "sexe_19",
    "Big_number_19",
    "comportement_19",
    "cellule_home_19",
    "cellule_work_19",
    "nb_records_19",
    "nb_moves_19",
]

#### Results

In [ ]:
# data = np.load(RESULTS_PATH / "merge_on_timeline_12_1769_by_19.npz", allow_pickle=True)
# data = np.load(RESULTS_PATH / "merge_on_timeline_filter_by_records_and_timestamps_12_1769_by_19.npz", allow_pickle=True)
data = np.load(RESULTS_PATH / "zero_and_best_matches_12_vs_19.npz", allow_pickle=True)
data_markov = np.load(RESULTS_PATH / "zero_and_best_matches_12_vs_19_markov_like.npz", allow_pickle=True)

# user_ids = data["user_ids"]
# distances = data["distances"]
# metas = data["metas"]

In [ ]:
data["results"]

In [ ]:
df = pd.json_normalize(data["results"])
df_markov = pd.json_normalize(data_markov["results"])

In [ ]:
df["best_nonzero_id_19"] = df["best_nonzero_id"].astype("Int64")
df = df.drop(columns = ["best_nonzero_id"])
df_markov["best_nonzero_id_19"] = df_markov["best_nonzero_id"].astype("Int64")
df_markov = df_markov.drop(columns = ["best_nonzero_id"])

In [ ]:
df = df.merge(
    df_meta_12,
    left_on="user_id",
    right_on="id_12",
    how="left",
    suffixes=("", "_user")
)
df = df.merge(
    df_meta_19,
    left_on="best_nonzero_id_19",
    right_on="id_19",
    how="left",
    suffixes=("", "_user")
)
df = df.drop(columns = ["user_id", "id_19"])
df_markov = df_markov.merge(
    df_meta_12,
    left_on="user_id",
    right_on="id_12",
    how="left",
    suffixes=("", "_user")
)
df_markov = df_markov.merge(
    df_meta_19,
    left_on="best_nonzero_id_19",
    right_on="id_19",
    how="left",
    suffixes=("", "_user")
)
df_markov = df_markov.drop(columns = ["user_id", "id_19"])

In [ ]:
df = df[
    [
        "zero_matches",
        "best_nonzero_distance",
        "id_12",
        "age_12",
        "sexe_12",
        "Big_number_12",
        "comportement_12",
        "cellule_home_12",
        "cellule_work_12",
        "nb_records_12",
        "nb_moves_12",
        "best_nonzero_id_19",
        "age_19",
        "sexe_19",
        "Big_number_19",
        "comportement_19",
        "cellule_home_19",
        "cellule_work_19",
        "nb_records_19",
        "nb_moves_12",
    ]
]
df_markov = df_markov[
    [
        "zero_matches",
        "best_nonzero_distance",
        "id_12",
        "age_12",
        "sexe_12",
        "Big_number_12",
        "comportement_12",
        "cellule_home_12",
        "cellule_work_12",
        "nb_records_12",
        "nb_moves_12",
        "best_nonzero_id_19",
        "age_19",
        "sexe_19",
        "Big_number_19",
        "comportement_19",
        "cellule_home_19",
        "cellule_work_19",
        "nb_records_19",
        "nb_moves_12",
    ]
]

In [ ]:
df

In [ ]:
df_markov

In [ ]:
df_events_19 = pd.DataFrame.from_dict(e_19, orient="index")

In [ ]:
df_events_19 = df_events_19.rename(columns={
    0: "cells",
    1: "timestamps"
})

In [ ]:
df_events_19["first_ts"] = df_events_19["timestamps"].apply(
    lambda x: x[0] if len(x) > 0 else None
)

df_events_19["last_ts"] = df_events_19["timestamps"].apply(
    lambda x: x[-1] if len(x) > 0 else None
)

In [ ]:
df_meta_19 = df_meta_19.merge(
    df_events_19[["first_ts", "last_ts"]],
    left_on="id_19",
    right_index=True,
    how="left"
)

In [ ]:
ids_zero = set()

for lst in df_markov["zero_matches"].dropna():
    ids_zero.update(lst)
    
ids_nonzero = set(df_markov["best_nonzero_id_19"].dropna())
ids_used = ids_zero.union(ids_nonzero)

ids_all = set(df_meta_19["id_19"])

In [ ]:
ids_missing = ids_all - ids_used
nb_missing = len(ids_missing)

In [ ]:
len(ids_zero)

In [ ]:
len(ids_nonzero)

In [ ]:
len(ids_used)

In [ ]:
df_markov_non_zero_matches = df[df["zero_matches"].str.len() == 0].drop(columns=["zero_matches"])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(
    data=df_markov_non_zero_matches,
    x="best_nonzero_distance",
    bins=40,
    kde=True  # ajoute une courbe de densité
)

plt.xlabel("Distance moyenne")
plt.ylabel("Nombre d'utilisateurs")
plt.title("Distribution des distances non nulles")
plt.grid(True)
plt.show()

In [ ]:
df_meta_19

In [ ]:
merged = df.merge(df_markov, on="id_12", suffixes=("_df", "_markov"))
similar_best = (
    merged["best_nonzero_id_19_df"] == merged["best_nonzero_id_19_markov"]
).mean()

In [ ]:
merged = merged.loc[:, ~merged.columns.duplicated()]

In [ ]:
merged

In [ ]:
similar_best = (
    merged["zero_matches_df"] == merged["zero_matches_markov"]
).mean()

In [ ]:
similar_best

In [ ]:
merged['similar'] = merged['zero_matches_df'] == merged['zero_matches_markov']

In [ ]:
sim = merged[['similar', 'zero_matches_df', 'zero_matches_markov']]

In [ ]:
merged

In [ ]:
print(merged.columns)

In [ ]:
filtered = merged[merged["nb_moves_12_df"] > 7]

In [ ]:
filtered